# SDA-DSC-212 — Time Series Analysis and Forecasting
## Lab 2 — The Stationarity Report

**Duration** 50 minutes · **Work in pairs** · `git checkout lab2-start`

> The one rule for this whole course: every feature, every split and every scaler
> must be computable using only data that existed at the forecast origin.

### Tasks

| min | task |
|---|---|
| 8 | Run `stationarity_report` on the raw log-demand for 2023. Interpret ADF and KPSS **together**. Record the verdict and predict the differencing you will need. |
| 10 | `choose_differencing` with `m=24` — seasonal difference first, then regular. Print the trace and re-run the report to confirm stationarity was reached. |
| 10 | Plot before / after differencing. Check the lag-1 ACF for the over-differencing signature. Reduce `d` if you see it. |
| 12 | ACF / PACF with `lags >= 60`. Annotate the seasonal spike at lag 24. Read candidate `(p,q)` and seasonal `(P,Q)` off the fingerprint rules. |
| 10 | Write `STATIONARITY.md`: the verdict, the chosen `(d, D)`, and 2–3 candidate SARIMA orders with justification. Commit. |

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})

DATA = "../data"

Lab 1's output, reloaded — the labs are independent notebooks.

In [ ]:
from tayyar.data.load import load_demand, load_calendar, join_calendar

df = load_demand(f"{DATA}/ksa_grid_demand.csv")
df = join_calendar(df, load_calendar(f"{DATA}/ksa_calendar.csv"))

y = df["demand_mw"].astype(float)
ylog = np.log(y)          # variance first (Lab 1), mean second (this lab)
sub = ylog["2023"]        # one clean year — the tests stay fast and the plots stay readable

print(f"log-demand, 2023: {len(sub):,} hourly observations")
print(f"from {sub.index[0]} to {sub.index[-1]}")

### Task 1 — ADF and KPSS, read together

### The two tests have opposite nulls

This is the single most common error in the room, so read it slowly.

| test | H0 (the null) | small p (< 0.05) means | large p means |
|---|---|---|---|
| **ADF** | there **is** a unit root — *non-stationary* | reject H0 → **evidence of stationarity** | cannot reject a unit root |
| **KPSS** | the series **is** stationary | reject H0 → **evidence of non-stationarity** | cannot reject stationarity |

A small p-value is good news from ADF and bad news from KPSS. Reconciling them:

| | KPSS says stationary | KPSS says non-stationary |
|---|---|---|
| **ADF says stationary** | `stationary` — proceed | `difference-stationary` — difference once and re-test |
| **ADF says non-stationary** | `trend-stationary` — detrend, or add a trend regressor | `non-stationary` — difference and re-test |

The disagreement cells are not a failure of the method. They are the reason we run both:
they separate a series that needs differencing from one that needs detrending.

In [ ]:
from tayyar.analysis.stationarity import stationarity_report, format_report

# TODO: run stationarity_report on `sub` and print it with format_report.
# Then write down, one sentence each:
#   1. what the ADF p-value ALONE would have you conclude,
#   2. what the KPSS p-value ALONE would have you conclude,
#   3. the reconciled verdict from the 2x2 table above,
#   4. your PREDICTION of (d, D) before Task 2 runs — commit to a number now.
#
# r0 = ...

# HINT: the nulls are opposite. Nearly every wrong answer in this room comes from
#       reading one test's p-value with the other test's null in mind.

> **Stop and predict.** You have a strongly seasonal hourly series with an annual swing.
> Write down the `(d, D)` you expect `choose_differencing` to return before you run the next
> cell. Most pairs write `d=1, D=1`. Keep your prediction — you will need it in a minute.

### Task 2 — the minimum differencing, seasonal first

In [ ]:
from tayyar.analysis.differencing import choose_differencing, difference

# TODO:
#   1. call choose_differencing(sub, m=24)
#   2. print the trace DataFrame — one row per differencing step, both p-values on each row
#   3. rebuild the differenced series with difference(sub, d, D, m=24)
#   4. re-run stationarity_report on it and confirm the verdict is "stationary"
#
# diffs = ...

# HINT: the routine takes the SEASONAL difference first and stops at the first
#       "stationary" verdict. Both of those are deliberate. Difference in the other
#       order, or keep going after the verdict flips, and you will spend a difference
#       you did not need — see Task 3 for what that costs.

### Task 3 — before / after, and the over-differencing check

In [ ]:
from statsmodels.tsa.stattools import acf

# TODO:
#   1. two stacked panels for June 2023: the level series, then the differenced series.
#      Add a zero line to the second panel — a stationary series should sit on it.
#   2. build the over-differencing ladder: for extra_d in (0, 1, 2), difference the series
#      with that many regular differences ON TOP of D=1, and report n, sd, variance and
#      the lag-1 ACF of each.
#   3. decide from the table whether the chosen (d, D) is the minimal one.
#
# HINT: the over-differencing signature is a lag-1 ACF strongly negative (below -0.5)
#       together with a variance that has gone UP, not down. A difference that lowers the
#       variance is not automatically a difference worth taking — the stationarity verdict
#       decides, not the variance.

### Task 4 — ACF and PACF: the model fingerprint

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from tayyar.analysis.correlograms import correlogram_table, suggest_orders, FINGERPRINT

# TODO:
#   1. a 2x2 correlogram figure with lags=72 — ACF and PACF of the LEVEL series (top row)
#      and of the differenced series (bottom row). Mark lag 24 on every panel.
#   2. correlogram_table(ds, nlags=72); print the rows at lags 1-5, 23, 24, 25, 48, 72.
#   3. suggest_orders(tbl, m=24), then read the fingerprint table and decide whether you
#      agree with what it returned.
#
# HINT: the default in every plotting API is lags=20, which NEVER REACHES LAG 24. Ask for
#       at least 2*m or the seasonal structure you came here to find is off the right-hand
#       edge of the plot. And treat suggest_orders as a candidate generator, never a verdict:
#       with n=8736 the significance band is +/-0.021, so almost everything is "significant".

### Mini-quiz — answer before you open the solution notebook

1. ADF returns p = 0.01. Stationary or not?
2. KPSS returns p = 0.01. Stationary or not?
3. For a seasonal series, which differencing comes first — seasonal or regular?
4. The ACF tails off and the PACF cuts off after lag 2. What model and order?
5. A slowly, almost linearly decaying ACF indicates what?

### Task 5 — `STATIONARITY.md`

Write the file. Four sections, no more:

1. **Verdict on the level series** — both p-values, and the reconciled verdict in words.
2. **Chosen `(d, D)` and `m`**, with the trace that justifies stopping there.
3. **Evidence that you did not over-difference** — the lag-1 ACF and the variance ladder.
4. **Two or three candidate SARIMA orders**, each with the correlogram feature that
   suggested it.

```
git add -A && git commit -m "feat(analysis): stationarity report + candidate SARIMA orders"
```